In [0]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import current_timestamp, to_utc_timestamp
from pyspark.sql.functions import when, col
from pyspark.sql.functions import *
from delta.tables import DeltaTable


In [0]:
dbutils.widgets.text("job_id", "")
job_id = dbutils.widgets.get("job_id")

In [0]:
watermark= spark.sql(f"""
SELECT watermark_column 
FROM pt.dev_metadata_schema.tbl_parameters 
where job_id='{job_id}' 
""")
watermark=watermark.collect()[0][0]

In [0]:
print(watermark)

In [0]:
fact_integcosting_df = spark.sql(f"""
        SELECT 
            *
        FROM pt.dev_silver.fact_integcosting
        WHERE to_timestamp(upd_date) > to_timestamp('{watermark}')
        """)

In [0]:
fact_integcosting_df=fact_integcosting_df.groupBy("project_id","tier_id",'currency','product_id').agg(sum("cost").alias('cost')).orderBy("project_id")

fact_integcosting_df=fact_integcosting_df.withColumns({'is_active':lit('Y'),'start_date':current_timestamp(),'end_date':lit(None)})

project_ids = [row.project_id for row in fact_integcosting_df.select("project_id").distinct().collect()]




In [0]:
if fact_integcosting_df.isEmpty():
        dbutils.notebook.exit("no_data")

In [0]:
gold_df = spark.createDataFrame([], fact_integcosting_df.schema)
if len(project_ids) >0:
    gold_df = spark.table("pt.dev_gold.fact_integcosting") \
        .select("project_id","tier_id","currency","product_id","cost","is_active","start_date","end_date")\
            .withColumns({"is_active":lit('N'),\
                         "end_date":current_timestamp()})\
    .filter(col("project_id").isin(project_ids))

final_df=fact_integcosting_df.unionAll(gold_df)

  

In [0]:
gold_table = DeltaTable.forName(spark, "pt.dev_gold.fact_integcosting")

In [0]:
gold_table.alias("t").merge(
    final_df.filter("is_active = 'N'").alias("s"),
    """
    t.project_id = s.project_id 
    AND t.tier_id = s.tier_id
    AND t.is_active = 'Y'
    and t.cost=s.cost
    """
).whenMatchedUpdate(
    set={
        "is_active": "'N'",
        "end_date": "s.end_date"
    }
).execute()

In [0]:

final_df.filter("is_active = 'Y'") \
    .write.format("delta") \
    .option("path", "abfss://gold@projecttrackingadlsdev.dfs.core.windows.net/fact_integcosting") \
    .mode("append") \
    .saveAsTable("pt.dev_gold.fact_integcosting")

In [0]:
spark.sql(f"""
UPDATE pt.dev_metadata_schema.tbl_parameters
SET watermark_column = current_timestamp
WHERE job_id='{job_id}'
""")